# Auditoría de preproceso HALCCON

Este notebook está diseñado para inspeccionar **paso a paso** el preproceso y detectar diferencias entre:

- datos crudos
- splits
- drops de columnas
- codificación del target
- CatBoost encoding de categóricas
- normalización Min-Max con estadísticas de `train`
- conversión final a tensores

El objetivo es que puedas ver **qué entra, qué sale y cómo cambian las estadísticas** en cada etapa.


## 0. Configuración

Ajusta estas rutas según el dataset y variante que quieras auditar.  
Por defecto está preparado para **UNSW-NB15 / `catboost_label__clean`**.


In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd
import torch

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 200)

# ========= AJUSTA SOLO ESTA SECCIÓN =========
DATASET_NAME = "unsw_nb15"
VARIANT_NAME = "catboost_label__clean"

RAW_TRAIN_CSV = Path("02_data/raw/data_unsw-nb15/UNSW_NB15_training-set.csv")
RAW_TEST_CSV  = Path("02_data/raw/data_unsw-nb15/UNSW_NB15_testing-set.csv")

SPLIT_DIR = Path(f"02_data/split/{DATASET_NAME}")
PROCESSED_DIR = Path(f"02_data/processed/{DATASET_NAME}/{VARIANT_NAME}")

TARGET_COL = "attack_cat"
DROP_COLS = ["id", "label"]   # attack_cat se separa como target
CAT_COLS = ["proto", "service", "state"]

print("DATASET_NAME:", DATASET_NAME)
print("VARIANT_NAME:", VARIANT_NAME)
print("RAW_TRAIN_CSV:", RAW_TRAIN_CSV)
print("RAW_TEST_CSV :", RAW_TEST_CSV)
print("SPLIT_DIR    :", SPLIT_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)


## 1. Cargar datos crudos y revisar esquema

In [ ]:

raw_train = pd.read_csv(RAW_TRAIN_CSV)
raw_test = pd.read_csv(RAW_TEST_CSV)

print("RAW TRAIN shape:", raw_train.shape)
print("RAW TEST  shape:", raw_test.shape)

print("\nRAW TRAIN columns:")
print(raw_train.columns.tolist())

print("\nRAW TEST columns:")
print(raw_test.columns.tolist())

print("\nMismas columnas en raw train/test:", list(raw_train.columns) == list(raw_test.columns))

print("\nDtypes RAW TRAIN:")
display(raw_train.dtypes.to_frame("dtype"))

print("\nDtypes RAW TEST:")
display(raw_test.dtypes.to_frame("dtype"))


## 2. Resumen rápido de calidad del dato

In [ ]:

def quick_quality_report(df, name):
    rep = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing_count": df.isna().sum(),
        "missing_pct": df.isna().mean() * 100,
        "nunique": df.nunique(dropna=False)
    }).sort_values(["missing_count", "nunique"], ascending=[False, False])
    print(f"=== {name} ===")
    display(rep)

quick_quality_report(raw_train, "RAW TRAIN")
quick_quality_report(raw_test, "RAW TEST")


## 3. Distribución del target crudo

In [ ]:

print("Target train:")
display(raw_train[TARGET_COL].value_counts(dropna=False).to_frame("count"))
display((raw_train[TARGET_COL].value_counts(normalize=True, dropna=False) * 100).to_frame("pct"))

print("Target test:")
display(raw_test[TARGET_COL].value_counts(dropna=False).to_frame("count"))
display((raw_test[TARGET_COL].value_counts(normalize=True, dropna=False) * 100).to_frame("pct"))


## 4. Cargar splits oficiales/intermedios si existen

In [ ]:

train_csv = SPLIT_DIR / "train.csv"
val_csv   = SPLIT_DIR / "val.csv"
test_csv  = SPLIT_DIR / "test.csv"

if train_csv.exists() and val_csv.exists() and test_csv.exists():
    split_train = pd.read_csv(train_csv)
    split_val   = pd.read_csv(val_csv)
    split_test  = pd.read_csv(test_csv)

    print("SPLIT TRAIN shape:", split_train.shape)
    print("SPLIT VAL   shape:", split_val.shape)
    print("SPLIT TEST  shape:", split_test.shape)
else:
    split_train = split_val = split_test = None
    print("No se encontraron train.csv / val.csv / test.csv en", SPLIT_DIR)


## 5. Funciones auxiliares de auditoría

In [ ]:

def numeric_stats(df, cols=None):
    use = df if cols is None else df[cols]
    out = pd.DataFrame({
        "min": use.min(numeric_only=True),
        "p01": use.quantile(0.01, numeric_only=True),
        "p25": use.quantile(0.25, numeric_only=True),
        "median": use.median(numeric_only=True),
        "mean": use.mean(numeric_only=True),
        "p75": use.quantile(0.75, numeric_only=True),
        "p99": use.quantile(0.99, numeric_only=True),
        "max": use.max(numeric_only=True),
        "std": use.std(numeric_only=True),
        "zero_pct": (use == 0).mean() * 100
    })
    return out

def compare_columns(df_a, df_b, name_a="A", name_b="B"):
    cols_a = list(df_a.columns)
    cols_b = list(df_b.columns)
    print(f"Mismas columnas {name_a}/{name_b}:", cols_a == cols_b)
    print(f"Solo en {name_a}:", sorted(set(cols_a) - set(cols_b)))
    print(f"Solo en {name_b}:", sorted(set(cols_b) - set(cols_a)))

def minmax_with_train_stats(X_train_df, X_other_df):
    train_min = X_train_df.min()
    train_max = X_train_df.max()
    denom = (train_max - train_min).replace(0, 1.0)
    X_train_scaled = (X_train_df - train_min) / denom
    X_other_scaled = (X_other_df - train_min) / denom
    return X_train_scaled, X_other_scaled, train_min, train_max, denom

def global_tensor_stats(x, name):
    arr = x.detach().cpu().numpy() if isinstance(x, torch.Tensor) else np.asarray(x)
    print(f"=== {name} ===")
    print("shape:", arr.shape)
    print("min:", np.nanmin(arr))
    print("max:", np.nanmax(arr))
    print("mean:", np.nanmean(arr))
    print("std:", np.nanstd(arr))
    print("has_nan:", np.isnan(arr).any())
    print("has_inf:", np.isinf(arr).any())


## 6. Separar `X` e `y`, aplicar drops y revisar columnas

In [ ]:

from sklearn.preprocessing import LabelEncoder

if split_train is None:
    raise RuntimeError("Necesitas tener train.csv / val.csv / test.csv para auditar el preproceso paso a paso.")

X_train = split_train.drop(columns=DROP_COLS + [TARGET_COL]).copy()
X_val   = split_val.drop(columns=DROP_COLS + [TARGET_COL]).copy()
X_test  = split_test.drop(columns=DROP_COLS + [TARGET_COL]).copy()

y_train_raw = split_train[TARGET_COL].copy()
y_val_raw   = split_val[TARGET_COL].copy()
y_test_raw  = split_test[TARGET_COL].copy()

print("X_train shape:", X_train.shape)
print("X_val   shape:", X_val.shape)
print("X_test  shape:", X_test.shape)

print("\nFeature columns:")
display(pd.DataFrame({"feature": X_train.columns}))

compare_columns(X_train, X_val, "train", "val")
compare_columns(X_train, X_test, "train", "test")

label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train_raw)
y_val   = label_encoder.transform(y_val_raw)
y_test  = label_encoder.transform(y_test_raw)

class_mapping = pd.DataFrame({
    "class_name": label_encoder.classes_,
    "class_id": range(len(label_encoder.classes_))
})
print("\nTarget mapping:")
display(class_mapping)


## 7. Estadísticas antes del encoding

In [ ]:

focus_cols = [c for c in [
    "proto", "service", "state",
    "dur", "rate", "sbytes", "dbytes",
    "sload", "dload", "sjit", "djit",
    "stcpb", "dtcpb",
    "tcprtt", "synack", "ackdat"
] if c in X_train.columns]

print("Focus cols:", focus_cols)
num_cols_pre = [c for c in focus_cols if c in X_train.select_dtypes(include=[np.number]).columns]
if num_cols_pre:
    display(numeric_stats(X_train.select_dtypes(include=[np.number]), cols=num_cols_pre))
else:
    print("No hay columnas numéricas en focus_cols antes del encoding.")


## 8. CatBoost encoding de categóricas

In [ ]:

from category_encoders import CatBoostEncoder

X_train_cb = X_train.copy()
X_val_cb   = X_val.copy()
X_test_cb  = X_test.copy()

for col in CAT_COLS:
    if col in X_train_cb.columns:
        X_train_cb[col] = X_train_cb[col].astype(str)
        X_val_cb[col]   = X_val_cb[col].astype(str)
        X_test_cb[col]  = X_test_cb[col].astype(str)

enc_cols = [c for c in CAT_COLS if c in X_train_cb.columns]
cb_encoder = CatBoostEncoder(
    cols=enc_cols,
    handle_unknown="value",
    handle_missing="value"
)

X_train_cb[enc_cols] = cb_encoder.fit_transform(X_train_cb[enc_cols], y_train)
X_val_cb[enc_cols]   = cb_encoder.transform(X_val_cb[enc_cols])
X_test_cb[enc_cols]  = cb_encoder.transform(X_test_cb[enc_cols])

print("Encoded shapes:")
print("X_train_cb:", X_train_cb.shape)
print("X_val_cb  :", X_val_cb.shape)
print("X_test_cb :", X_test_cb.shape)

print("\nObject columns after encoding:")
print("train:", X_train_cb.select_dtypes(include=["object"]).columns.tolist())
print("val  :", X_val_cb.select_dtypes(include=["object"]).columns.tolist())
print("test :", X_test_cb.select_dtypes(include=["object"]).columns.tolist())

display(X_train_cb.head())


## 9. Estadísticas después del encoding y antes de normalizar

In [ ]:

focus_post = [c for c in focus_cols if c in X_train_cb.columns]
display(numeric_stats(X_train_cb, cols=focus_post))
display(numeric_stats(X_test_cb, cols=focus_post))


## 10. Normalización Min-Max con estadísticas de train (igual al pipeline `clean`)

In [ ]:

X_train_scaled, X_val_scaled, train_min, train_max, denom = minmax_with_train_stats(X_train_cb, X_val_cb)
X_train_scaled_2, X_test_scaled, train_min2, train_max2, denom2 = minmax_with_train_stats(X_train_cb, X_test_cb)

assert train_min.equals(train_min2)
assert train_max.equals(train_max2)
assert denom.equals(denom2)
assert X_train_scaled.equals(X_train_scaled_2)

global_tensor_stats(X_train_scaled.values, "X_train_scaled")
global_tensor_stats(X_val_scaled.values, "X_val_scaled")
global_tensor_stats(X_test_scaled.values, "X_test_scaled")


## 11. Features fuera de rango en val/test tras Min-Max

In [ ]:

range_check = pd.DataFrame({
    "train_min_scaled": X_train_scaled.min(),
    "train_max_scaled": X_train_scaled.max(),
    "val_min_scaled": X_val_scaled.min(),
    "val_max_scaled": X_val_scaled.max(),
    "test_min_scaled": X_test_scaled.min(),
    "test_max_scaled": X_test_scaled.max(),
})

print("Features with val_min < 0:")
display(range_check[range_check["val_min_scaled"] < 0])

print("Features with val_max > 1:")
display(range_check[range_check["val_max_scaled"] > 1].sort_values("val_max_scaled", ascending=False))

print("Features with test_min < 0:")
display(range_check[range_check["test_min_scaled"] < 0])

print("Features with test_max > 1:")
display(range_check[range_check["test_max_scaled"] > 1].sort_values("test_max_scaled", ascending=False))


## 12. Conversión a tensores

In [ ]:

X_train_tensor = torch.tensor(X_train_scaled.values, dtype=torch.float32).unsqueeze(1)
X_val_tensor   = torch.tensor(X_val_scaled.values, dtype=torch.float32).unsqueeze(1)
X_test_tensor  = torch.tensor(X_test_scaled.values, dtype=torch.float32).unsqueeze(1)

y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_val_tensor   = torch.tensor(y_val, dtype=torch.long)
y_test_tensor  = torch.tensor(y_test, dtype=torch.long)

global_tensor_stats(X_train_tensor, "X_train_tensor")
global_tensor_stats(X_val_tensor, "X_val_tensor")
global_tensor_stats(X_test_tensor, "X_test_tensor")

print("Tensor shapes:")
print("X_train:", X_train_tensor.shape, "y_train:", y_train_tensor.shape)
print("X_val  :", X_val_tensor.shape, "y_val  :", y_val_tensor.shape)
print("X_test :", X_test_tensor.shape, "y_test :", y_test_tensor.shape)


## 13. Comparación opcional con artefactos ya guardados por el pipeline

In [ ]:

artifacts = {
    "X_train.pt": PROCESSED_DIR / "X_train.pt",
    "X_val.pt": PROCESSED_DIR / "X_val.pt",
    "X_test.pt": PROCESSED_DIR / "X_test.pt",
    "y_train.pt": PROCESSED_DIR / "y_train.pt",
    "y_val.pt": PROCESSED_DIR / "y_val.pt",
    "y_test.pt": PROCESSED_DIR / "y_test.pt",
}

missing = [k for k, v in artifacts.items() if not v.exists()]
if missing:
    print("Faltan artefactos:", missing)
else:
    saved = {k: torch.load(v) for k, v in artifacts.items()}
    for k, v in saved.items():
        global_tensor_stats(v, f"saved::{k}")

    print("\nComparación exacta con tensores recreados:")
    print("X_train equal:", torch.equal(saved["X_train.pt"], X_train_tensor))
    print("X_val   equal:", torch.equal(saved["X_val.pt"], X_val_tensor))
    print("X_test  equal:", torch.equal(saved["X_test.pt"], X_test_tensor))
    print("y_train equal:", torch.equal(saved["y_train.pt"], y_train_tensor))
    print("y_val   equal:", torch.equal(saved["y_val.pt"], y_val_tensor))
    print("y_test  equal:", torch.equal(saved["y_test.pt"], y_test_tensor))


## 14. Checklist de auditoría

Usa esta sección para clasificar hallazgos:

- **equivalente al notebook / pipeline**
- **divergente**
- **posible bug**
- **ambigüedad**
- **mejora propuesta**

Preguntas clave:
1. ¿Las columnas finales y su orden coinciden?
2. ¿El target mapping coincide?
3. ¿Las categorías codificadas coinciden?
4. ¿La normalización usa solo estadísticas de train?
5. ¿Los tensores finales son exactamente iguales a los artefactos guardados?
6. ¿Hay features fuera de rango en val/test después del Min-Max?
7. ¿Hay señales de compresión excesiva por outliers?
